# On-Device Real Estate Assistant — Flan-T5 Fine-Tuning 


## 1. Environment Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os, random, json
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    T5ForConditionalGeneration,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
import evaluate

SEED       = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME  = "google/flan-t5-base"
MAX_INPUT   = 512
MAX_TARGET  = 256
BATCH_SIZE  = 16
EPOCHS      = 20
LR          = 3e-4
OUTPUT_DIR  = "./flan_t5_zillow"

print("Config ready.")

## 2. Load & Explore Dataset

In [ ]:
raw_dataset = load_dataset("zillow/real_estate_v1")
print(raw_dataset)

example = raw_dataset['train'][1]
print(f"\nTopic: {example['topic']}")
for msg in example['messages'][:4]:
    print(f"[{msg['role'].upper()}]: {msg['content'][:200]}")

## 3. Data Processing

In [ ]:
def extract_qa_pairs(row):
    messages = row['messages']
    topic    = row['topic']
    pairs    = []

    if not messages or len(messages) < 2:
        return pairs

    for i in range(len(messages) - 1):
        cur = messages[i]
        nxt = messages[i + 1]

        if cur['role'] != 'user' or nxt['role'] != 'assistant':
            continue

        user_msg = cur['content'].strip()
        asst_msg = nxt['content'].strip()

        if len(asst_msg) < 50 or len(user_msg) < 10:
            continue

        prior = messages[max(0, i - 2):i]
        context_str = ""
        if prior:
            parts = []
            for m in prior:
                role = "Customer" if m['role'] == 'user' else "Agent"
                parts.append(f"{role}: {m['content'][:200].strip()}")
            context_str = "\n".join(parts)

        if context_str:
            input_text = (
                f"You are a helpful real estate assistant.\n"
                f"Topic: {topic}\n"
                f"Previous context:\n{context_str}\n"
                f"Customer question: {user_msg}\n"
                f"Answer:"
            )
        else:
            input_text = (
                f"You are a helpful real estate assistant.\n"
                f"Topic: {topic}\n"
                f"Customer question: {user_msg}\n"
                f"Answer:"
            )

        pairs.append({
            "input_text":  input_text,
            "target_text": asst_msg,
            "topic":       topic,
        })
    return pairs


print("Extracting QA pairs...")
all_pairs = []
for row in raw_dataset['train']:
    all_pairs.extend(extract_qa_pairs(row))
print(f"Total QA pairs: {len(all_pairs)}")

# Train / Val / Test split
random.shuffle(all_pairs)
n      = len(all_pairs)
n_val  = int(n * 0.10)
n_test = int(n * 0.10)
test_pairs  = all_pairs[:n_test]
val_pairs   = all_pairs[n_test:n_test + n_val]
train_pairs = all_pairs[n_test + n_val:]
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")

def pairs_to_dataset(pairs):
    return Dataset.from_dict({
        "input_text":  [p['input_text']  for p in pairs],
        "target_text": [p['target_text'] for p in pairs],
    })

dataset = DatasetDict({
    "train": pairs_to_dataset(train_pairs),
    "val":   pairs_to_dataset(val_pairs),
    "test":  pairs_to_dataset(test_pairs),
})
print(dataset)

In [ ]:
# ── DIAGNOSTIC 1: Check raw extracted pairs ──
print("=== Sample Target Texts ===")
for i in range(5):
    print(f"[{i}] INPUT:  {train_pairs[i]['input_text'][:100]}")
    print(f"[{i}] TARGET: {train_pairs[i]['target_text'][:200]}")
    print()

In [ ]:
# ── DIAGNOSTIC 2: Check tokenized labels ──
print("=== Tokenized Label Check ===")
for i in range(3):
    label_ids = tokenized["train"][i]["labels"]
    decoded   = tokenizer.decode(label_ids, skip_special_tokens=True)
    print(f"[{i}] Label ids (first 10): {label_ids[:10]}")
    print(f"[{i}] Decoded label:        {decoded[:200]}")
    print()

In [ ]:
# ── DIAGNOSTIC 3: Check if labels are all identical ──
labels_0 = tokenized["train"][0]["labels"]
labels_1 = tokenized["train"][1]["labels"]
labels_2 = tokenized["train"][2]["labels"]
print("Label 0 == Label 1:", labels_0 == labels_1)
print("Label 0 == Label 2:", labels_0 == labels_2)

In [ ]:
# ── DIAGNOSTIC 4: Check what model actually generates vs reference ──
sample = tokenized["val"][0]
input_ids = torch.tensor([sample["input_ids"]]).cuda() if torch.cuda.is_available() else torch.tensor([sample["input_ids"]])
with torch.no_grad():
    out = model.generate(input_ids, max_new_tokens=100)
generated = tokenizer.decode(out[0], skip_special_tokens=True)
reference = tokenizer.decode(
    [x for x in sample["labels"] if x != -100],
    skip_special_tokens=True
)
print("GENERATED: ", generated)
print("REFERENCE: ", reference)

## 4. Tokenization
**Key fix here:** we use `text_target` argument (not `as_target_tokenizer`) and explicitly avoid storing any `-100` values in tokenized outputs — those are only applied by the `DataCollatorForSeq2Seq` at batch time, which is the safe place for them.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize(batch):
    # Tokenize inputs separately from targets
    model_inputs = tokenizer(
        text=batch["input_text"],
        max_length=MAX_INPUT,
        truncation=True,
        padding=False,
    )

    # Tokenize targets separately with their own max_length
    labels = tokenizer(
        text=batch["target_text"],
        max_length=MAX_TARGET,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing...")
tokenized = dataset.map(
    tokenize,
    batched=True,
    batch_size=512,
    remove_columns=["input_text", "target_text"],
)

# Sanity check — no negative ids (those only appear later via DataCollator)
sample_labels = tokenized["train"][0]["labels"]
assert all(x >= 0 for x in sample_labels), "Negative values found in labels!"
print("Sanity check passed.")
print(tokenized)

## 5. Compute Metrics
**Key fix here:** replace `-100` with `pad_token_id` before calling `batch_decode`, which prevents the OverflowError during evaluation.

In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Handle tuple output from model
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    vocab_size = tokenizer.vocab_size
    predictions = np.clip(predictions, 0, vocab_size - 1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels,      skip_special_tokens=True)

    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # Guard against empty predictions
    if not any(decoded_preds):
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )
    return {
        "rouge1": round(result["rouge1"], 4),
        "rouge2": round(result["rouge2"], 4),
        "rougeL": round(result["rougeL"], 4),
    }

## 6. Load Model & Train

In [ ]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME, from_tf=True)
if torch.cuda.is_available():
    model = model.cuda()

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=5,  
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.05, 
    lr_scheduler_type="cosine",

    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET,

    logging_dir="./logs",
    logging_steps=20,
    report_to="none",

    fp16=False,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),  
    dataloader_num_workers=2,
    gradient_accumulation_steps=4,
    seed=SEED,
)

In [ ]:
from transformers import EarlyStoppingCallback, TrainerCallback

class GradientMonitorCallback(TrainerCallback):
    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step == 5:   # check after 5 steps
            total_grad = 0
            for name, param in model.named_parameters():
                if param.grad is not None:
                    total_grad += param.grad.abs().sum().item()
            print(f"\nStep 5 total gradient norm: {total_grad:.6f}")
            if total_grad == 0:
                print("WARNING: Zero gradients — weights are not updating!")
            else:
                print("Gradients flowing correctly.")
            control.should_training_stop = False  # don't stop, just report

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["val"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        GradientMonitorCallback(),
        EarlyStoppingCallback(early_stopping_patience=2)
    ],
)

In [ ]:
# ── Train ──
print("Starting training...")
train_result = trainer.train()
print(f"\nDone! Loss: {train_result.training_loss:.4f}")
print(f"Steps:       {train_result.global_step}")
print(f"Runtime:     {train_result.metrics.get('train_runtime', 0):.0f}s")

# ── Save ──
SAVE_PATH = "./flan_t5_zillow_final2"
os.makedirs(SAVE_PATH, exist_ok=True)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"\nModel saved to: {SAVE_PATH}")
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f"  {f:<40} {size:.1f} MB")

In [ ]:
# ── Per-epoch ROUGE summary ──
print("\n=== Epoch ROUGE Summary ===")
for log in trainer.state.log_history:
    if "eval_rougeL" in log:
        epoch = log.get("epoch", "?")
        print(f"  Epoch {epoch:.0f} — ROUGE-L: {log['eval_rougeL']:.4f} | ROUGE-1: {log.get('eval_rouge1', 0):.4f}")

In [ ]:
# ── Save ──
SAVE_PATH = "./flan_t5_zillow_final"
os.makedirs(SAVE_PATH, exist_ok=True)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"\nModel saved to: {SAVE_PATH}")
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f"  {f:<40} {size:.1f} MB")

## 7. Evaluate on Test Set

In [ ]:
print("Evaluating on test set...")
test_results = trainer.predict(
    tokenized["test"],
    max_length=MAX_TARGET,
    num_beams=4,
)
m = test_results.metrics
print(f"ROUGE-1: {m.get('test_rouge1', 'N/A')}")
print(f"ROUGE-2: {m.get('test_rouge2', 'N/A')}")
print(f"ROUGE-L: {m.get('test_rougeL', 'N/A')}  (target >= 0.35)")

In [ ]:
# Interactive test
def generate_answer(question, topic, context=""):
    if context:
        prompt = (
            f"You are a helpful real estate assistant.\n"
            f"Topic: {topic}\nPrevious context:\n{context}\n"
            f"Customer question: {question}\nAnswer:"
        )
    else:
        prompt = (
            f"You are a helpful real estate assistant.\n"
            f"Topic: {topic}\nCustomer question: {question}\nAnswer:"
        )
    inputs = tokenizer(prompt, return_tensors="pt", max_length=MAX_INPUT, truncation=True)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, num_beams=4, no_repeat_ngram_size=3)
    return tokenizer.decode(out[0], skip_special_tokens=True)

tests = [
    ("What are the steps to get pre-approved for a mortgage?", "Home Financing"),
    ("How do I make a competitive offer in a seller's market?", "Buying Process"),
    ("What should I look for during a home inspection?",       "Home Inspections"),
]
for q, t in tests:
    print(f"Q [{t}]: {q}")
    print(f"A: {generate_answer(q, t)}\n")

In [ ]:
def generate_answer(question, topic=None, context=""):
    # Build topic line only if provided
    topic_line = f"Topic: {topic}\n" if topic else ""

    if context:
        prompt = (
            f"You are an expert real estate agent assistant. "
            f"Give a thorough, detailed answer to the customer's question.\n\n"
            f"{topic_line}"
            f"Previous context:\n{context}\n"
            f"Question: {question}\n\n"
            f"Provide a comprehensive, step-by-step answer with examples:"
        )
    else:
        prompt = (
            f"You are an expert real estate agent assistant. "
            f"Give a thorough, detailed answer to the customer's question.\n\n"
            f"{topic_line}"
            f"Question: {question}\n\n"
            f"Provide a comprehensive, step-by-step answer with examples:"
        )

    inputs = tokenizer(prompt, return_tensors="pt", max_length=MAX_INPUT, truncation=True)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            min_new_tokens=100,
            num_beams=4,
            no_repeat_ngram_size=4,
            repetition_penalty=2.0,
            length_penalty=2.0,
            early_stopping=False,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

## 8. Save Model

In [ ]:
SAVE_PATH = "./flan_t5_zillow_final"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Saved to {SAVE_PATH}")
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f"  {f:<40} {size:.1f} MB")